In [ ]:
# !pip install pandas --quiet
# !pip install seaborn --quiet
# !pip install  scikit-learn --quiet
# !pip install matplotlib --quiet
# !pip install plotly --quiet
# !pip install plotly jupyterlab --quiet                                                 
# !pip install networkx --quiet 

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
from sklearn.ensemble import IsolationForest

pyo.init_notebook_mode()
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/sensors.csv")
print(df.shape)

In [ ]:
print("\n=== Пропуски ===")
print(df.isnull().sum())

In [ ]:
df = df.rename(columns={
    'Timestamp': 'timestamp',
    'car-id':    'car_id',
    'car-type':  'car_type',
    'gate-name': 'gate_name',
})

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['car_id']  = df['car_id'].astype('string')
df['car_type']  = df['car_type'].astype('category')
df['gate_name'] = df['gate_name'].astype('category')

print(df.dtypes)

In [ ]:
print("\n=== Уникальные значения ===")
print(f"Машин:         {df['car_id'].nunique()}")
print(f"Типов машин:   {df['car_type'].nunique()} → {sorted(df['car_type'].unique().tolist())}")
print(f"Ворот:         {df['gate_name'].nunique()} → {sorted(df['gate_name'].unique().tolist())}")
print(f"Период:        {df['timestamp'].min()}  →  {df['timestamp'].max()}")
print(f"Всего событий: {len(df)}")

In [ ]:
df = df.sort_values(['car_id', 'timestamp']).reset_index(drop=True)

# Временны́е признаки
df['date']        = df['timestamp'].dt.date
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()       
df['day_of_week_n']= df['timestamp'].dt.dayofweek      
df['is_weekend']  = df['day_of_week_n'].isin([5, 6])
df['month']       = df['timestamp'].dt.month
df['week']        = df['timestamp'].dt.isocalendar().week.astype(int)

# Маршруты: следующие/предыдущие ворота 
df['prev_gate'] = df.groupby('car_id')['gate_name'].shift(1)
df['next_gate'] = df.groupby('car_id')['gate_name'].shift(-1)

# Время между событиями одной машины 
df['time_since_prev'] = (
    df.groupby('car_id')['timestamp']
      .diff()                          
)

# Порядковый номер визита машины
df['visit_order'] = df.groupby('car_id').cumcount() + 1

print(df.head(10))

In [ ]:
pivot = df.pivot_table(index='hour', columns='day_of_week', 
                       values='car_id', aggfunc='count')

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = pivot.reindex(columns=[d for d in days_order if d in pivot.columns])

sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='d')
plt.title('Активность по часам и дням недели')

In [ ]:
# По воротам и часам
pivot_gate_hour = df.pivot_table(
    index='gate_name', columns='hour',
    values='car_id', aggfunc='count'
)
sns.heatmap(pivot_gate_hour, cmap='Blues', annot=True, fmt='.0f')
plt.title('Активность: ворота × час')

In [ ]:
ts = (df.groupby([pd.Grouper(key='timestamp', freq='h'), 'gate_name'])
        .size()
        .reset_index(name='count'))

fig = px.line(
    ts, x='timestamp', y='count', color='gate_name',
    title='Трафик через каждые ворота (по часам)',
    labels={'count': 'Кол-во проездов', 'timestamp': 'Время'},
    template='plotly_white'
)
fig.update_layout(hovermode='x unified')
fig.show()

# Отдельно — с выделением выходных
daily = df.groupby(['date', 'is_weekend']).size().reset_index(name='count')
daily['date'] = pd.to_datetime(daily['date'])

fig2 = px.bar(
    daily, x='date', y='count',
    color=daily['is_weekend'].map({True: 'Выходной', False: 'Будний'}),
    color_discrete_map={'Выходной': '#e74c3c', 'Будний': '#3498db'},
    title='Дневной трафик: будние vs выходные',
    labels={'color': 'Тип дня'}
)
fig2.show()

In [ ]:
# Строим граф переходов между воротами
edges = (df.dropna(subset=['prev_gate'])
           .groupby(['prev_gate', 'gate_name'])
           .size()
           .reset_index(name='weight'))

G = nx.DiGraph()
for _, row in edges.iterrows():
    G.add_edge(row['prev_gate'], row['gate_name'], weight=row['weight'])

# --- Визуализация через NetworkX ---
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=2)

weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w   = max(weights)

nx.draw_networkx_nodes(G, pos, node_size=1500, node_color='#3498db', alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=9, font_color='black', font_weight='bold')
nx.draw_networkx_edges(
    G, pos,
    width=[2 + 6 * w / max_w for w in weights],
    alpha=0.7,
    edge_color=weights,
    edge_cmap=plt.cm.Reds,
    arrows=True, arrowsize=20,
    connectionstyle='arc3,rad=0.1'
)
edge_labels = {(u, v): d['weight'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)

plt.title('Граф переходов между воротами', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('gate_graph.png', dpi=150)
plt.show()

In [ ]:
# Top-N переходов для читаемости
top_edges = edges.nlargest(30, 'weight')

all_nodes = list(pd.unique(top_edges[['prev_gate', 'gate_name']].values.ravel()))
node_idx  = {n: i for i, n in enumerate(all_nodes)}

fig = go.Figure(go.Sankey(
    node=dict(
        label=all_nodes,
        pad=15, thickness=20,
        color='#3498db'
    ),
    link=dict(
        source=[node_idx[r] for r in top_edges['prev_gate']],
        target=[node_idx[r] for r in top_edges['gate_name']],
        value=top_edges['weight'].tolist(),
        color='rgba(52, 152, 219, 0.4)'
    )
))
fig.update_layout(
    title_text='Sankey: потоки между воротами',
    font_size=12, height=600
)
fig.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 3.1 Распределение типов машин по воротам
ct = df.groupby(['gate_name', 'car_type']).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, ax=axes[0, 0],
            colormap='tab10', edgecolor='white')
axes[0, 0].set_title('Состав типов машин по воротам (%)')
axes[0, 0].set_xlabel('')
axes[0, 0].legend(title='Тип', bbox_to_anchor=(1, 1))
axes[0, 0].tick_params(axis='x', rotation=45)

# 3.2 Активность типов машин по часам
hour_type = df.groupby(['hour', 'car_type']).size().unstack(fill_value=0)
hour_type.plot(kind='area', stacked=True, ax=axes[0, 1],
               colormap='tab10', alpha=0.8)
axes[0, 1].set_title('Активность типов машин по часам суток')
axes[0, 1].set_xlabel('Час')
axes[0, 1].legend(title='Тип', bbox_to_anchor=(1, 1))

# 3.3 Boxplot: количество визитов на машину по типу
visits_per_car = df.groupby(['car_id', 'car_type']).size().reset_index(name='visits')
sns.boxplot(data=visits_per_car, x='car_type', y='visits',
            palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Кол-во визитов на машину по типу')
axes[1, 0].tick_params(axis='x', rotation=45)

# 3.4 Тепловая карта: тип машины × ворота (нормировано)
hm = df.groupby(['car_type', 'gate_name']).size().unstack(fill_value=0)
hm_norm = hm.div(hm.sum(axis=1), axis=0)
sns.heatmap(hm_norm, cmap='Blues', annot=True, fmt='.0f', ax=axes[1, 1])
axes[1, 1].set_title('Доля посещений: тип машины × ворота')

plt.tight_layout()
plt.savefig('car_type_patterns.png', dpi=150)
plt.show()

In [ ]:
# --- 4.1 Топ машин по количеству визитов ---
top_cars = df['car_id'].value_counts().head(20).reset_index()
top_cars.columns = ['car_id', 'visits']

fig = px.bar(top_cars, x='car_id', y='visits',
             title='Топ-20 машин по числу проездов',
             color='visits', color_continuous_scale='Reds')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# --- 4.2 Аномалии через Isolation Forest ---
features = df.groupby('car_id').agg(
    total_visits   = ('car_id',         'count'),
    unique_gates   = ('gate_name',      'nunique'),
    avg_interval_s = ('time_since_prev', lambda x: x.dt.total_seconds().mean()),
    visit_hours_std= ('hour',           'std'),
).fillna(0)

iso = IsolationForest(contamination=0.05, random_state=42)
features['anomaly'] = iso.fit_predict(features)
features['anomaly_label'] = features['anomaly'].map({1: 'Normal', -1: 'Anomaly'})

fig2 = px.scatter(
    features.reset_index(),
    x='total_visits', y='unique_gates',
    color='anomaly_label',
    hover_name='car_id',
    size='avg_interval_s',
    color_discrete_map={'Normal': '#3498db', 'Anomaly': '#e74c3c'},
    title='Аномальные машины (Isolation Forest)',
    labels={'total_visits': 'Всего проездов', 'unique_gates': 'Уник. ворот'}
)
fig2.show()

print("\nАномальные машины:")
print(features[features['anomaly'] == -1].sort_values('total_visits', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 5.1 Распределение интервалов между проездами
intervals_min = df['time_since_prev'].dt.total_seconds().dropna() / 60
intervals_clip = intervals_min.clip(upper=intervals_min.quantile(0.95))

axes[0].hist(intervals_clip, bins=60, color='#3498db', edgecolor='white')
axes[0].set_title('Распределение интервалов между проездами (мин, до 95-го перцентиля)')
axes[0].set_xlabel('Минуты')
axes[0].set_ylabel('Частота')

# 5.2 Средний интервал по воротам
avg_interval = (df.groupby('gate_name')['time_since_prev']
                  .apply(lambda x: x.dt.total_seconds().mean() / 60))
avg_interval.sort_values().plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Средний интервал до следующего проезда по воротам (мин)')
axes[1].set_xlabel('Минуты')

plt.tight_layout()
plt.savefig('intervals.png', dpi=150)
plt.show()

In [ ]:
flow_counts = df.groupby(['gate_name', 'next_gate']).size().reset_index(name='count')

gates = list(set(flow_counts['gate_name'].tolist() + flow_counts['next_gate'].tolist()))
gate_idx = {g: i for i, g in enumerate(gates)}

fig = go.Figure(go.Sankey(
    node=dict(label=gates),
    link=dict(
        source=[gate_idx[s] for s in flow_counts['gate_name']],
        target=[gate_idx[t] for t in flow_counts['next_gate']],
        value=flow_counts['count']
    )
))
fig.show()

In [ ]:
G = nx.DiGraph()
for _, row in flow_counts.iterrows():
    G.add_edge(row['gate_name'], row['next_gate'], weight=row['count'])

weights = [G[u][v]['weight'] * 0.01 for u, v in G.edges()]
pos = nx.spring_layout(G, seed=42)

nx.draw_networkx(G, pos, width=weights, node_size=1000,
                 node_color='steelblue', font_color='white',
                 arrows=True, arrowsize=20)
plt.title('Граф переходов между воротами')
plt.show()

In [ ]:
hourly_rank = (df.groupby(['hour', 'gate_name'])
                 .size()
                 .reset_index(name='count'))
hourly_rank['rank'] = hourly_rank.groupby('hour')['count'].rank(ascending=False)

fig = px.line(hourly_rank, x='hour', y='rank', color='gate_name',
              title='Рейтинг ворот по загруженности в течение дня',
              markers=True)
fig.update_yaxes(autorange='reversed')
fig.show()

In [ ]:
hourly = df.groupby(['hour', 'car_type']).size().reset_index(name='count')

fig = px.bar_polar(hourly, r='count', theta='hour',
                   color='car_type', template='plotly_dark',
                   title='Суточная активность по типу машин')
fig.show()

In [ ]:
# !pip install holoviews bokeh --quiet
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')

matrix = pd.crosstab(df['gate_name'], df['next_gate'])
chord = hv.Chord(matrix)
chord.opts(opts.Chord(cmap='Category20', edge_color='source', 
                      node_color='index', labels='name'))
hv.save(chord, 'chord.html')

In [ ]:
sample_cars = df['car_id'].value_counts().head(20).index
df_sample = df[df['car_id'].isin(sample_cars)].copy()
df_sample['end'] = df_sample['timestamp'] + pd.Timedelta(minutes=5)

fig = px.timeline(df_sample, x_start='timestamp', x_end='end',
                  y='car_id', color='gate_name',
                  title='Маршруты машин во времени')
fig.show()

In [ ]:
traffic_ts = df.set_index('timestamp').resample('1h')['car_id'].count()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, lag in enumerate([1, 24, 168]):  # 1ч, 1 день, 1 неделя
    axes[i].scatter(traffic_ts[:-lag], traffic_ts[lag:], alpha=0.4, s=10)
    axes[i].set_title(f'Lag = {lag}h')
    axes[i].set_xlabel('t'); axes[i].set_ylabel('t + lag')
plt.suptitle('Lag Plots — поиск периодичности')
plt.show()

In [ ]:
from sklearn.ensemble import IsolationForest

# Признаки: час + ворота (encoded) + тип машины (encoded)
features = pd.get_dummies(df[['hour', 'gate_name', 'car_type']])
iso = IsolationForest(contamination=0.02, random_state=42)
df['anomaly'] = iso.fit_predict(features)

fig = px.scatter(df, x='timestamp', y='gate_name',
                 color=df['anomaly'].map({1: 'normal', -1: 'anomaly'}),
                 title='Аномалии в данных',
                 color_discrete_map={'normal': 'steelblue', 'anomaly': 'red'})
fig.show()